# B2-022-probabilistic-latent-models — Practice p10 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** autoencoder

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `aio-ml:F1-scientific-python`, `aio-ml:F4-matrices`, `aio-ml:F3-multivar-calculus`, `aio-ml:F5-probability`, `aio-ml:F6-svd-spectral`, `aio-ml:C1-ml-fundamentals`, `aio-ml:C2-linear-models`, `aio-ml:C6-neural-networks`, `aio-ml:C7-pytorch`, `aio-ml:C9-dimensionality-reduction`, `aio-ml:C8-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [aio-ml:F1-scientific-python](../../../../aio-ml/units/01-F1-scientific-python/lesson.ipynb), [aio-ml:F4-matrices](../../../../aio-ml/units/04-F4-matrices/lesson.ipynb), [aio-ml:F3-multivar-calculus](../../../../aio-ml/units/03-F3-multivar-calculus/lesson.ipynb), [aio-ml:F5-probability](../../../../aio-ml/units/05-F5-probability/lesson.ipynb), [aio-ml:F6-svd-spectral](../../../../aio-ml/units/12-F6-svd-spectral/lesson.ipynb), [aio-ml:C1-ml-fundamentals](../../../../aio-ml/units/06-C1-ml-fundamentals/lesson.ipynb), [aio-ml:C2-linear-models](../../../../aio-ml/units/07-C2-linear-models/lesson.ipynb), [aio-ml:C6-neural-networks](../../../../aio-ml/units/11-C6-neural-networks/lesson.ipynb), [aio-ml:C7-pytorch](../../../../aio-ml/units/13-C7-pytorch/lesson.ipynb), [aio-ml:C9-dimensionality-reduction](../../../../aio-ml/units/16-C9-dimensionality-reduction/lesson.ipynb), [aio-ml:C8-neural-training](../../../../aio-ml/units/15-C8-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch
from torch import nn

SEED = 20260927
x = torch.arange(30, dtype=torch.float32).reshape(5, 6) / 10.0

## Solution

The modules are created in the pinned order, with no activation after the code layer or the output layer.

**Parameter count** (weights + biases):

| layer | weights | biases | total |
|---|---|---|---|
| `encoder[0] = Linear(6,12)` | $6\cdot12=72$ | $12$ | $84$ |
| `encoder[2] = Linear(12,3)` | $12\cdot3=36$ | $3$ | $39$ |
| `decoder[0] = Linear(3,12)` | $3\cdot12=36$ | $12$ | $48$ |
| `decoder[2] = Linear(12,6)` | $12\cdot6=72$ | $6$ | $78$ |
| **total** | | | $84+39+48+78=\mathbf{249}$ |

In [ ]:
class MLPAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(6, 12), nn.ReLU(), nn.Linear(12, 3))
        self.decoder = nn.Sequential(nn.Linear(3, 12), nn.ReLU(), nn.Linear(12, 6))

    @staticmethod
    def _check(t, last, name):
        if t.dim() != 2 or t.shape[-1] != last:
            raise ValueError(f"{name} must be 2-D with last dimension {last}, got {tuple(t.shape)}")

    def encode(self, x):
        self._check(x, 6, "x")
        return self.encoder(x)

    def decode(self, h):
        self._check(h, 3, "h")
        return self.decoder(h)

    def forward(self, x):
        self._check(x, 6, "x")
        return self.decode(self.encode(x))


def shape_ledger(model, x):
    enc_hidden = model.encoder[1](model.encoder[0](x))
    code = model.encoder[2](enc_hidden)
    dec_hidden = model.decoder[1](model.decoder[0](code))
    recon = model.decoder[2](dec_hidden)
    return [("input", tuple(x.shape)), ("encoder_hidden", tuple(enc_hidden.shape)),
            ("code", tuple(code.shape)), ("decoder_hidden", tuple(dec_hidden.shape)),
            ("reconstruction", tuple(recon.shape))]


torch.manual_seed(SEED)
model = MLPAutoencoder()
ledger = shape_ledger(model, x)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
y = model(x)

def raises_value_error(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

# torch.zeros: tensor of zeros
bad_cases = [
    raises_value_error(lambda: model(torch.zeros(5, 7))),
    raises_value_error(lambda: model.encode(torch.zeros(5, 7))),
    raises_value_error(lambda: model(torch.zeros(6))),
    raises_value_error(lambda: model.encode(torch.zeros(6))),
    raises_value_error(lambda: model.decode(torch.zeros(5, 2))),
]
print(ledger, n_params)

### Answer check

In [ ]:
# Checks: the autoencoder answers against pinned values
assert ledger == [("input", (5, 6)), ("encoder_hidden", (5, 12)), ("code", (5, 3)),
                  ("decoder_hidden", (5, 12)), ("reconstruction", (5, 6))]
assert model.encoder[0].weight.shape == (12, 6)
assert model.encoder[2].weight.shape == (3, 12)
assert model.decoder[0].weight.shape == (12, 3)
assert model.decoder[2].weight.shape == (6, 12)
assert n_params == 249
assert torch.equal(model(x), model.decode(model.encode(x)))
assert y.dtype == torch.float32
assert len(model.encoder) == 3 and len(model.decoder) == 3
assert isinstance(model.encoder[-1], nn.Linear) and isinstance(model.decoder[-1], nn.Linear)
assert all(bad_cases)